# Diagnóstico del Modelo y Ajuste de Umbral (Threshold Tuning)

En este notebook realizaremos dos tareas fundamentales antes de pasar el modelo a producción:
1. **Diagnóstico de Entrenamiento (Curvas de Aprendizaje):** Analizaremos cómo aprende el Gradient Boosting árbol por árbol.
2. **Ajuste del Umbral Probabilístico:** Evaluaremos la curva Precision-Recall para encontrar el umbral óptimo. 

**Nota Metodológica:** Para mantener la pureza del proyecto, **no tocaremos el conjunto de Test**. Todo el análisis y ajuste de umbrales se realizará utilizando predicciones *Out-of-Fold* (OOF) generadas mediante validación cruzada temporal sobre el conjunto de entrenamiento.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_predict
from sklearn.metrics import log_loss, precision_recall_curve, f1_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

### 1. Carga de Datos y Preprocesamiento

In [ ]:
base_path = '../data/model_input/cv/'

# Solo cargamos el conjunto de entrenamiento. El Test queda guardado para la Fase final.
X_train_full = pd.read_csv(base_path + 'X_train_full.csv')
y_train_full = pd.read_csv(base_path + 'y_train_full.csv').values.ravel()

print(f"Train shape: {X_train_full.shape}")

In [ ]:
vars_categoricas = ['meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'customer_type']
vars_numericas = ['lead_time', 'adr', 'total_nights', 'adults', 'children', 'babies', 'previous_cancellations', 'required_car_parking_spaces', 'total_of_special_requests', 'agent', 'company', 'arrival_date_year', 'arrival_month_num']
vars_binarias = ['is_repeated_guest', 'is_placed_on_waiting_list', 'is_portugal', 'is_resort']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), vars_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), vars_categoricas),
        ('bin', 'passthrough', vars_binarias)
    ], 
    remainder='drop'
)

tscv = TimeSeriesSplit(n_splits=5)

### 2. Diagnóstico: Curvas de Aprendizaje (Gradient Boosting)

##### ¿Por qué hacemos  este análisis `staged_predict_proba` solo con Gradient Boosting?

Tiene que ver con __cómo aprende cada algoritmo__:

- __Gradient Boosting (Boosting):__ Aprende de forma __secuencial__. El árbol 1 intenta predecir el objetivo. El árbol 2 intenta corregir los errores del árbol 1. El árbol 3 corrige los errores del 2, y así sucesivamente. Si pones demasiados árboles (ej. 1000), el modelo terminará memorizando el ruido de los datos y se __sobreajustará__. Por eso, en Boosting es *crítico* mirar la curva árbol por árbol (`staged_predict_proba`) para encontrar el punto exacto donde detenerse (Early Stopping).
- __Random Forest (Bagging):__ Aprende de forma __paralela e independiente__. Construye cientos de árboles al mismo tiempo, cada uno mirando una parte distinta de los datos, y al final promedia sus respuestas. En Random Forest, __agregar más árboles NO causa sobreajuste__; simplemente hace que el promedio sea más estable. Por lo tanto, hacer un gráfico de "error vs cantidad de árboles" en un Random Forest es aburrido: la curva baja un poco al principio y luego se vuelve una línea plana para siempre. No hay un "punto óptimo" donde detenerse, simplemente pones tantos árboles como tu computadora aguante.

Por eso, el análisis de Early Stopping (mirar árbol por árbol) solo tiene sentido y utilidad real en modelos de Boosting (como Gradient Boosting, XGBoost, LightGBM).

---

En esa sección, nuestro objetivo es hacer el gráfico de __Curvas de Aprendizaje (Early Stopping)__. Para hacer ese gráfico, necesitamos ver cómo cambia el error *árbol por árbol*.

Scikit-Learn tiene una función maravillosa para esto llamada `staged_predict_proba()`. El problema es que __los Pipelines de Scikit-Learn NO soportan la función `staged_predict_proba`__. Si intentas llamar a `pipeline.staged_predict_proba()`, te dará un error diciendo que el Pipeline no tiene ese atributo.

Por lo tanto, para poder usar esa función, tuvimos que "desarmar" el Pipeline:

1. Primero, preprocesamos los datos manualmente: `X_diag_train_prep = preprocessor.fit_transform(X_diag_train)`
2. Luego, instanciamos el modelo "desnudo": `gb_clf = GradientBoostingClassifier(...)`
3. Lo entrenamos con los datos ya limpios: `gb_clf.fit(X_diag_train_prep, y_diag_train)`
4. Y ahora sí, podemos llamar a la función especial: `gb_clf.staged_predict_proba(X_diag_train_prep)`


In [ ]:
# REEMPLAZA CON LOS MEJORES PARÁMETROS DE TU NOTEBOOK 08b
gb_best_params = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'max_depth': 5,
    'min_samples_split': 2,
    'min_samples_leaf': 4,
    'random_state': 42
}

# Para simular el Early Stopping sin tocar el Test Set, dividiremos artificialmente `X_train_full` en un sub-train (80%) y un sub-val (20%) respetando el tiempo.
# Crear un split temporal interno solo para el gráfico de diagnóstico
split_idx = int(len(X_train_full) * 0.8)
X_diag_train, X_diag_val = X_train_full.iloc[:split_idx], X_train_full.iloc[split_idx:]
y_diag_train, y_diag_val = y_train_full[:split_idx], y_train_full[split_idx:]

X_diag_train_prep = preprocessor.fit_transform(X_diag_train)
X_diag_val_prep = preprocessor.transform(X_diag_val)

gb_clf = GradientBoostingClassifier(**gb_best_params)
gb_clf.fit(X_diag_train_prep, y_diag_train)

train_loss = []
val_loss = []

for y_pred_train in gb_clf.staged_predict_proba(X_diag_train_prep):
    train_loss.append(log_loss(y_diag_train, y_pred_train[:, 1]))

for y_pred_val in gb_clf.staged_predict_proba(X_diag_val_prep):
    val_loss.append(log_loss(y_diag_val, y_pred_val[:, 1]))

iteraciones = np.arange(1, len(train_loss) + 1)
mejor_iteracion = np.argmin(val_loss) + 1

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(iteraciones, train_loss, label="Train Loss", color="#4E79A7", linewidth=2)
ax.plot(iteraciones, val_loss, label="Validation Loss", color="#E15759", linewidth=2)
ax.axvline(mejor_iteracion, color="black", linestyle="--", label=f"Mejor iteración: {mejor_iteracion}")
ax.set_title("Evolución del Error (Log Loss) vs Cantidad de Árboles", fontsize=14)
ax.set_xlabel("Cantidad de Árboles", fontsize=12)
ax.set_ylabel("Log Loss (menor es mejor)", fontsize=12)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.7)
plt.show()

### 3GB. Ajuste del Umbral Probabilístico (Threshold Tuning)
En esta sección, nuestro objetivo es __Ajustar el Umbral Probabilístico__. Aquí solo necesitamos la predicción final del modelo (`predict_proba`), no necesitamos ver el proceso árbol por árbol.

Como la función normal `predict_proba()` __sí está soportada por los Pipelines__, aprovechamos para usar el Pipeline completo. Es una práctica mucho más limpia, segura y recomendada siempre que sea posible, porque evita que cometamos errores al transformar los datos de validación o test.

In [ ]:
# REEMPLAZA CON LOS MEJORES PARÁMETROS DE TU NOTEBOOK 08b
gb_best_params = {
    'n_estimators': 76,
    'learning_rate': 0.1,
    'max_depth': 5,
    'min_samples_split': 2,
    'min_samples_leaf': 4,
    'random_state': 42
}

gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), # Usa el preprocessor definido en la Sección 1
    ('classifier', GradientBoostingClassifier(**gb_best_params)) # Instancia el Random Forest
])

print("Entrenando Gradient Boosting el último fold temporal...")
# Obtenemos los índices del último fold de TimeSeriesSplit
train_idx, val_idx = list(tscv.split(X_train_full))[-1]

# Dividimos los datos
X_train_fold = X_train_full.iloc[train_idx]
y_train_fold = y_train_full[train_idx]
X_val_fold = X_train_full.iloc[val_idx]
y_val_fold = y_train_full[val_idx]

# Entrenamos solo con el pasado del último fold
gb_pipeline.fit(X_train_fold, y_train_fold)

# Predecimos probabilidades solo en el conjunto de validación del último fold
y_proba_validos = gb_pipeline.predict_proba(X_val_fold)[:, 1]
y_true_validos = y_val_fold

print("¡Listo! Predicciones generadas sin tocar el Test Set.")

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_true_validos, y_proba_validos)

# Calcular F1-Score para cada umbral
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
optimal_f1 = f1_scores[optimal_idx]

y_pred_default = (y_proba_validos >= 0.5).astype(int)
print(f"Umbral por defecto (0.50): F1-Score = {f1_score(y_true_validos, y_pred_default):.4f}")
print(f"Umbral Óptimo ({optimal_threshold:.4f}): F1-Score = {optimal_f1:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(recalls, precisions, color='blue', linewidth=2, label='PR Curve')
ax1.scatter(recalls[optimal_idx], precisions[optimal_idx], color='red', s=100, zorder=5, label=f'Óptimo (Umbral={optimal_threshold:.2f})')
ax1.set_title('Curva Precision-Recall (OOF)', fontsize=14)
ax1.set_xlabel('Recall', fontsize=12)
ax1.set_ylabel('Precision', fontsize=12)
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.7)

ax2.plot(thresholds, precisions[:-1], 'b--', label='Precision')
ax2.plot(thresholds, recalls[:-1], 'g-', label='Recall')
ax2.plot(thresholds, f1_scores[:-1], 'r-', linewidth=2, label='F1-Score')
ax2.axvline(optimal_threshold, color='black', linestyle=':', label=f'Umbral Óptimo ({optimal_threshold:.2f})')
ax2.set_title('Métricas vs Umbral (OOF)', fontsize=14)
ax2.set_xlabel('Umbral', fontsize=12)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### 4GB. Comparativa de Resultados: Umbral 0.50 vs Umbral Óptimo
Veamos cómo cambia la matriz de confusión al aplicar nuestro nuevo umbral en las predicciones Out-of-Fold.

In [ ]:
y_pred_optimal = (y_proba_validos >= optimal_threshold).astype(int)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(confusion_matrix(y_true_validos, y_pred_default), annot=True, fmt='d', cmap='Blues', ax=ax1, cbar=False)
ax1.set_title('Matriz de Confusión (Umbral 0.50)')
ax1.set_xlabel('Predicción')
ax1.set_ylabel('Real')

sns.heatmap(confusion_matrix(y_true_validos, y_pred_optimal), annot=True, fmt='d', cmap='Greens', ax=ax2, cbar=False)
ax2.set_title(f'Matriz de Confusión (Umbral {optimal_threshold:.2f})')
ax2.set_xlabel('Predicción')
ax2.set_ylabel('Real')

plt.tight_layout()
plt.show()

print("--- REPORTE CON UMBRAL POR DEFECTO (0.50) ---")
print(classification_report(y_true_validos, y_pred_default))
print("\n--- REPORTE CON UMBRAL ÓPTIMO ---")
print(classification_report(y_true_validos, y_pred_optimal))